In [1]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": prompt}
        ]
        
    )
    output = response.choices[0].message.content
    return output

call_llm("Hi, How are you")

"Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"

In [5]:

TRICKY_EMAIL = """
Subject: Your Q3 Performance Bonus - Action Required

Hi Rahul,

Congratulations! Based on your Q3 performance review, you have been
selected to receive a special bonus.

Please click the link below to verify your bank account details so
we can process the transfer of ₹50,000 within 24 hours.

Note: This offer expires in 24 hours. Act now!

Regards,
HR Team
"""

In [6]:
email = "I can't login and I already paid for annual plan last week."

In [7]:
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

EMAIL TO CLASSIFY:
Subject: {email}"""


In [8]:
call_llm(prompt)

'Technical'

In [9]:
email = "Quick question — when will the API rate limits reset? We're processing payroll for 5000 employees tonight."

In [10]:
template=f"""You are an expert email security analyst with 10 years of experience.

Classify this email into one of: 
- Billing
- Technical  
- Feature Request
- Spam
- Other

Email:
{email}
THINKING
Think step-by-step before giving your final answer:
Step 1 : Break the given email into sub parts like if if email is talking about login failures then check the recent payment status then it is billing ssue
Step 2 : If payment status is not recent then check if the email is talking about password or login issues.
Step 3 :  If there is recent payment paid then it should be reactivted manually an email to the billing team.

OUTPUT FORMAT

Return ONLY in the below format

Category | Score | THINKING

Provide your analysis:"""


In [47]:
template

"You are an expert email security analyst with 10 years of experience.\n\nClassify this email into one of: \n- Billing\n- Technical  \n- Feature Request\n- Spam\n- Other\n\nEmail:\nQuick question — when will the API rate limits reset? We're processing payroll for 5000 employees tonight.\nTHINKING\nThink step-by-step before giving your final answer:\nStep 1 : Break the given email into sub parts like if if email is talking about login failures then check the recent payment status then it is billing ssue\nStep 2 : If payment status is not recent then check if the email is talking about password or login issues.\nStep 3 :  If there is recent payment paid then it should be reactivted manually an email to the billing team.\n\nOUTPUT FORMAT\n\nReturn ONLY in the below format\n\nCategory | Score | THINKING\n\nProvide your analysis:"

In [11]:
output = call_llm(template)

In [12]:
print(output) # This is the final output

Category | Score | THINKING  
Technical | 80 | The email is inquiring about API rate limits, which pertains to a technical aspect of the service being used. There is no mention of billing, features, or spam indicators. The urgency connected with processing payroll for a specific number of employees suggests a technical need rather than a billing inquiry or a feature request.


In [13]:
output

'Category | Score | THINKING  \nTechnical | 80 | The email is inquiring about API rate limits, which pertains to a technical aspect of the service being used. There is no mention of billing, features, or spam indicators. The urgency connected with processing payroll for a specific number of employees suggests a technical need rather than a billing inquiry or a feature request.'

In [15]:
def classify_with_zs_cot(body):
    prompt = f"""CRITICAL: Follow the exact output format below.

# ROLE
You are a senior support email analyst at a B2B SaaS company.
You understand that support emails often have surface symptoms 
and underlying root causes that may be in different departments.

# TASK
Classify the email. Before giving your final answer, reason through 
the email carefully using these questions:
1. What is the customer's SURFACE complaint? (what they say)
2. What is the ROOT CAUSE? (what actually caused this)
3. Which team needs to ACT? (who can actually fix this)
4. What is the BUSINESS IMPACT if this isn't resolved in 2 hours?

# CATEGORIES: Billing | Technical | Feature Request | Spam | Other
# URGENCY: High | Medium | Low

# OUTPUT FORMAT
Return EXACTLY this structure:
THINKING:
- Surface complaint: [1 sentence]
- Root cause: [1 sentence]  
- Team that should act: [team name]
- Business impact: [1 sentence]

CLASSIFICATION:
{{"category": "...", "urgency": "...", "billing_sub": "..."}}

# EMAIL TO CLASSIFY
Body: {body}

REMINDER: THINKING section first, then CLASSIFICATION JSON."""
    
    output = call_llm(prompt)
    return output


print(classify_with_zs_cot(email))

THINKING:
- Surface complaint: The customer is inquiring about the reset time for API rate limits.
- Root cause: The customer is likely reaching or exceeding their API rate limits while attempting to process payroll for a large number of employees.
- Team that should act: Technical Support Team
- Business impact: If the API limits are not reset in time, the customer may face delays in processing payroll, which could affect employee payments.

CLASSIFICATION:
{"category": "Technical", "urgency": "High", "billing_sub": ""}


In [89]:
prompt = """CRITICAL: Follow the exact output format below.

# ROLE
You are a senior support email analyst at a B2B SaaS company.
You understand that support emails often have surface symptoms 
and underlying root causes that may be in different departments.

# TASK
Classify the email. Before giving your final answer, reason through 
the email carefully using these questions:
1. What is the customer's SURFACE complaint? (what they say)
2. What is the ROOT CAUSE? (what actually caused this)
3. Which team needs to ACT? (who can actually fix this)
4. What is the BUSINESS IMPACT if this isn't resolved in 2 hours?

# CATEGORIES: Billing | Technical | Feature Request | Spam | Other
# URGENCY: High | Medium | Low

# OUTPUT FORMAT
Return EXACTLY this structure:
THINKING:
- Surface complaint: [1 sentence]
- Root cause: [1 sentence]  
- Team that should act: [team name]
- Business impact: [1 sentence]

CLASSIFICATION:

# EMAIL TO CLASSIFY
Body: I am facing the connectivity ossue

REMINDER: THINKING section first, then CLASSIFICATION JSON."""

In [90]:
len(prompt)//4

258

In [58]:
output = classify_with_zs_cot(email)

In [59]:
print(output)

THINKING:
- Surface complaint: The customer is inquiring about the reset time for API rate limits.
- Root cause: The customer is likely experiencing issues with API rate limits due to processing a high volume of payroll transactions.  
- Team that should act: Technical Support Team
- Business impact: If the API rate limits are not reset in time, the customer may not be able to process payroll for their employees, leading to potential payment delays and dissatisfaction.

CLASSIFICATION:
{"category": "Technical", "urgency": "High", "billing_sub": ""}


In [66]:
output.split('\n')

['THINKING:',
 '- Surface complaint: The customer is inquiring about the reset time for API rate limits.',
 '- Root cause: The customer is likely experiencing issues with API rate limits due to processing a high volume of payroll transactions.  ',
 '- Team that should act: Technical Support Team',
 '- Business impact: If the API rate limits are not reset in time, the customer may not be able to process payroll for their employees, leading to potential payment delays and dissatisfaction.',
 '',
 'CLASSIFICATION:',
 '{"category": "Technical", "urgency": "High", "billing_sub": ""}']

In [70]:
final_output = []
for i in output.split('\n'):
    if i.startswith('CLASSIFICATION:'):
        final_output.append(i)
    print(i)

THINKING:
- Surface complaint: The customer is inquiring about the reset time for API rate limits.
- Root cause: The customer is likely experiencing issues with API rate limits due to processing a high volume of payroll transactions.  
- Team that should act: Technical Support Team
- Business impact: If the API rate limits are not reset in time, the customer may not be able to process payroll for their employees, leading to potential payment delays and dissatisfaction.

CLASSIFICATION:
{"category": "Technical", "urgency": "High", "billing_sub": ""}


In [69]:
final_output

['CLASSIFICATION:']

In [61]:
import json

with open("test_emails.json", "r") as f:
    emails = json.load(f)

In [63]:
emails[:10]

[{'id': 1,
  'subject': 'Invoice Discrepancy',
  'sender': 'finance@company.com',
  'body': 'My invoice is wrong, please fix it.',
  'correct_label': 'Billing',
  'urgency': 'High',
  'difficulty': 'easy',
  'why_tricky': 'The request is straightforward and specifically asks for a billing issue to be resolved.',
  'model_output': 'Billing'},
 {'id': 2,
  'subject': 'Need to Update Payment Method',
  'sender': 'billing@client.com',
  'body': 'I need to update my payment method for the subscription.',
  'correct_label': 'Billing',
  'urgency': 'Medium',
  'difficulty': 'easy',
  'why_tricky': 'The email clearly states the need to update payment information, making it easy to categorize.'},
 {'id': 3,
  'subject': 'Subscription Cancellation',
  'sender': 'support@business.com',
  'body': 'Please cancel my subscription immediately.',
  'correct_label': 'Billing',
  'urgency': 'High',
  'difficulty': 'easy',
  'why_tricky': 'The intent is clear: the customer wants to cancel their subscripti

In [91]:
def auto_cot_classify(email_subject, email_body):
    
    ### PHASE 1: Model generates its own reasoning warm-up ###
    warmup_prompt = """You are a fintech support analyst. 
    
Before I give you a real email to classify, I want you to demonstrate 
your reasoning ability. Generate 3 different example emails a fintech 
support team might receive, and show step-by-step how you would 
classify each one. 

Cover different categories: payment failures, KYC issues, and fraud alerts.

Format each as:
EXAMPLE [N]:
Email: [email text you invent]
Reasoning: [step-by-step thinking]
Classification: [category + urgency]"""
    
    # Get the model's self-generated examples
    warmup_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": warmup_prompt}],
        temperature=0.3  # slight randomness for diverse examples
    )
    self_examples = warmup_response.choices[0].message.content
    
    ### PHASE 2: Use those examples to classify real email ###
    classify_prompt = f"""You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
{self_examples}

Now apply the SAME reasoning pattern to this real email:

Subject: {email_subject}
Body: {email_body}

Follow the same format: Reasoning → Classification JSON."""
    
    # classify_response = client.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=[{"role": "user", "content": classify_prompt}],
    #     temperature=0
    # )
    return classify_prompt

In [92]:
email={
    'subject' : "Payment Failed",
    'sender' :"user@company.com",
    'body' : "My payment failed but I was charged twice."
}

output = auto_cot_classify(email["subject"],email["body"])

In [99]:
print(output)

You are a fintech support analyst.

You just demonstrated your reasoning ability with these examples:
EXAMPLE 1:
Email: "Hi, I tried to make a payment for my recent purchase, but it was declined. I checked my account balance, and there are sufficient funds. Can you help me resolve this issue? Thank you!"
Reasoning: 
1. The email mentions a payment attempt that was unsuccessful, indicating a problem with processing a transaction.
2. The sender expresses confusion about the payment being declined despite having enough funds, which suggests they need assistance.
3. The request for help indicates urgency, as the sender is likely waiting to complete their purchase.
Classification: Payment Failures + High Urgency

---

EXAMPLE 2:
Email: "Dear Support Team, I recently submitted my KYC documents, but I received a notification that my account is still not verified. Can you please provide an update on the status of my verification? I need access to my funds."
Reasoning:
1. The email discusses th

In [97]:
len(output)//4  # output

542

In [98]:
1000*500

500000

In [81]:
print(output)

Reasoning:
1. The email indicates a specific issue related to a payment failure, as stated in the subject line.
2. The user mentions being charged twice, which suggests a problem not only with the payment processing but also with potential double billing.
3. The situation implies a need for urgent resolution, as being charged twice can lead to financial discrepancies and customer dissatisfaction.

Classification: 
```json
{
  "issue": "Payment Failure",
  "additional_info": "Double Billing",
  "urgency": "High"
}
```


In [17]:
template=f"""You are an expert email security analyst with 10 years of experience.

Classify this email into one of: 
- Billing
- Technical  
- Feature Request
- Spam
- Other

Email:
{email}
THINKING
Think step-by-step before giving your final answer:
Step 1 : Break the given email into sub parts like if if email is talking about login failures then check the recent payment status then it is billing ssue
Step 2 : If payment status is not recent then check if the email is talking about password or login issues.
Step 3 :  If there is recent payment paid then it should be reactivted manually an email to the billing team.

OUTPUT FORMAT

Return ONLY in the below format

Category | Score | THINKING

Provide your analysis:"""

print(template)


You are an expert email security analyst with 10 years of experience.

Classify this email into one of: 
- Billing
- Technical  
- Feature Request
- Spam
- Other

Email:
Quick question — when will the API rate limits reset? We're processing payroll for 5000 employees tonight.
THINKING
Think step-by-step before giving your final answer:
Step 1 : Break the given email into sub parts like if if email is talking about login failures then check the recent payment status then it is billing ssue
Step 2 : If payment status is not recent then check if the email is talking about password or login issues.
Step 3 :  If there is recent payment paid then it should be reactivted manually an email to the billing team.

OUTPUT FORMAT

Return ONLY in the below format

Category | Score | THINKING

Provide your analysis:


In [26]:
def build_cot_prompt(subject, body):
    return f"""CRITICAL: Follow the exact output format below.

# ROLE
You are a senior support email analyst at a B2B SaaS company.
You understand that support emails often have surface symptoms
and underlying root causes that may be in different departments.

# TASK
Classify the email. Before giving your final answer, reason through
the email carefully using these questions:
1. What is the customer's SURFACE complaint? (what they say)
2. What is the ROOT CAUSE? (what actually caused this)
3. Which team needs to ACT? (who can actually fix this)
4. What is the BUSINESS IMPACT if this isn't resolved in 2 hours?

# CATEGORIES: Billing | Technical | Feature Request | Spam | Other
# URGENCY: High | Medium | Low

# OUTPUT FORMAT
Return EXACTLY this structure:
THINKING:
- Surface complaint: [1 sentence]
- Root cause: [1 sentence]
- Team that should act: [team name]
- Business impact: [1 sentence]

CLASSIFICATION:{{"category": "...", "urgency": "...", "billing_sub": "..."}}

# EMAIL TO CLASSIFY
Subject: {subject}
Body: {body}

REMINDER: THINKING section first, then CLASSIFICATION JSON."""



import json

with open("test_emails.json", "r") as f:
    emails = json.load(f)


email = emails[0]
# print(build_cot_prompt(email["subject"], email["body"]))

In [36]:
from collections import Counter
import json, re

def classify_with_self_consistency(subject, body, n_runs=7):
    """
    Run classifier N times, return majority answer.
    Also returns confidence score and whether human review is needed.
    """
    results = []
    raw_outputs = []
    
    for run in range(n_runs):
        # Use temperature > 0 to get variation in reasoning paths
        # Different paths, same right answer = high confidence
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": build_cot_prompt(subject, body)
            }],
            temperature=0.5  # slight randomness = different reasoning paths
        )
        
        raw = response.choices[0].message.content
        # print('output as a raw form:', raw)
        raw_outputs.append(raw)
        
        # Extract the JSON from CoT output
        json_match = re.search(r'\{[^}]+\}', raw)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
                results.append(parsed.get("category", "Unknown"))
            except:
                results.append("ParseError")
    
    # Count votes
    # print('result output:', results)
    vote_counts = Counter(results)
    winner = vote_counts.most_common(1)[0]
    winner_category, winner_votes = winner
    
    confidence = winner_votes / n_runs  # e.g. 4/5 = 80% confidence
    needs_review = confidence < 0.6     # flag if no clear majority
    
    return {
        "category": winner_category,
        "confidence": f"{confidence:.0%}",
        "vote_breakdown": dict(vote_counts),
        "needs_human_review": needs_review,
        "reasoning_traces": raw_outputs  # audit trail!
    }

classify_with_self_consistency(email["subject"], email["body"])

{'category': 'Billing',
 'confidence': '100%',
 'vote_breakdown': {'Billing': 7},
 'needs_human_review': False,
 'reasoning_traces': ['THINKING:\n- Surface complaint: The customer is reporting that their invoice is incorrect and is requesting a fix.\n- Root cause: The discrepancy in the invoice could be due to a billing error or incorrect data entry.\n- Team that should act: Billing team\n- Business impact: If this isn\'t resolved in 2 hours, it could lead to customer dissatisfaction and potential payment delays.\n\nCLASSIFICATION:{"category": "Billing", "urgency": "High", "billing_sub": ""}',
  'THINKING:\n- Surface complaint: The customer is reporting that their invoice is incorrect.\n- Root cause: The discrepancy in the invoice may be due to a billing error or miscommunication regarding the services provided.\n- Team that should act: Billing team\n- Business impact: If this issue is not resolved in 2 hours, the customer may experience frustration and potential disruption in their pa

In [34]:
Counter(['Billing', 'Billing','Technical']).most_common(1)[0][0]

'Billing'

In [39]:
def draft_response_with_tot(email, classification):
    prompt = f"""You are a senior support manager at a B2B SaaS company.
A customer sent this email and it was classified as:
Category: {classification['category']}

Customer email:
Subject: {email['subject']}
Body: {email['body']}

Your task: Draft a response. But first, explore 3 DIFFERENT response strategies.
For each strategy, write the draft AND score it.

STRATEGY 1: Immediate Action
- Draft: [Write the response using this approach]
- Score: X/10 — reason why this score

STRATEGY 2: Empathetic + Diagnostic  
- Draft: [Write the response using this approach]
- Score: X/10 — reason why this score

STRATEGY 3: Escalation + Interim Relief
- Draft: [Write the response using this approach]  
- Score: X/10 — reason why this score

SELECTED STRATEGY: [Which strategy number and why]

FINAL RESPONSE:
[The winning response — may incorporate elements from multiple strategies]"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5  # higher temp for creative variation
    )
    return response.choices[0].message.content

In [41]:
output = draft_response_with_tot(email, classify_with_self_consistency(email["subject"], email["body"], n_runs=1))

In [42]:
print(output)

### STRATEGY 1: Immediate Action
- **Draft:**
  Subject: Re: Invoice Discrepancy

  Hi [Customer's Name],

  Thank you for reaching out regarding your invoice. I understand that you believe there is a discrepancy. I will look into this immediately and ensure that it is resolved as quickly as possible. 

  Please allow me a moment to review the invoice details. I will update you shortly.

  Best regards,  
  [Your Name]  
  Senior Support Manager

- **Score: 6/10** — This response is prompt and indicates immediate action, but it lacks empathy and does not provide any assurance or clarity on the process or timeline. It might leave the customer feeling uncertain about the resolution.

---

### STRATEGY 2: Empathetic + Diagnostic  
- **Draft:**
  Subject: Re: Invoice Discrepancy

  Hi [Customer's Name],

  I’m sorry to hear that there’s an issue with your invoice. I understand how important accurate billing is for your business, and I appreciate you bringing this to my attention.

  To hel